In [7]:
#Instalamos la libreria de STREAMLIT
%pip install streamlit


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [8]:
#Instalamos la libreria de PLOTLY
%pip install plotly


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [17]:
%%writefile app.py 
#Creamos el archivo de la APP en el interprete principal (Phyton)
#####################################################
#Importamos librerias
import streamlit as st
import plotly.express as px
import pandas as pd
import numpy as np
from scipy.optimize import curve_fit
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import statsmodels.api as sm
from statsmodels.formula.api import ols
from scipy import stats
######################################################
#Definimos la instancia
@st.cache_resource
######################################################
#Creamos la función de carga de datos
def load_data():
   #Lectura del archivo csv
   df = pd.read_excel("Datos_CLV.xlsx", sheet_name="Histórico_Ventas")
    #Rellenamos nulos
   df =df.bfill()
   df =df.ffill()  
   #Creamos la lista de las variables categóricas sobre las que se generará la extracción de características(análisis univariado)
   Lista=['Survived', 'Pclass', 'Sex', 'Age']
   return df, Lista
###############################################################################
#Cargo los datos obtenidos de la función "load_data"
df, Lista = load_data()
###############################################################################
#CREACIÓN DEL DASHBOARD
#Generamos las páginas que utilizaremos en el diseño
##############################################################################
#Generamos los encabezados para la barra lateral (sidebar)
st.sidebar.title("DATOS CLV")

#Widget 1: Selectbox
#Menu desplegable de opciones de laa páginas seleccionadas
View= st.sidebar.selectbox(label= "Tipo de Análisis", options= ["Extracción de Características", 
                                              "Regresión Lineal", "Regresión No Lineal",
                                              "Regresión Logística", "ANOVA"])

 # CONTENIDO DE LA VISTA 5
if View == "ANOVA":

    # Variables de la base
    Lista_num = [
        'precio_unitari_o',
        'total_unidades_de_producto',
        'total_neto_deproducto'
    ]

    Lista_cat = [
        'marca_',
        'ciudad_mx',
        'vendedor_EWS'
    ]

    # Seleccion de variables
    Variable_y = st.sidebar.selectbox(
        label="Variable dependiente (y)",
        options=Lista_num
    )

    Variable_x = st.sidebar.selectbox(
        label="Variable independiente (x)",
        options=Lista_cat
    )

    st.title("ANOVA")
    cantidad = st.sidebar.slider(
        "Cantidad de grupos",
        min_value=2,
        max_value=df[Variable_x].nunique(),
        value=2
    )

    seleccionados = sorted(df[Variable_x].dropna().unique())[:cantidad]
    df = df[df[Variable_x].isin(seleccionados)]

    Contenedor_A, Contenedor_B = st.columns(2)

    # Resultados del ANOVA
    with Contenedor_A:
        st.subheader("Resultados del ANOVA")

        grupos = [
            grupo[Variable_y].dropna()
            for nombre, grupo in df.groupby(Variable_x)
        ]

        if len(grupos) >= 2 and all(len(grupo) >= 2 for grupo in grupos):
            f_valor, p_valor = stats.f_oneway(*grupos)

            st.write(f"**F-valor:** {f_valor:.3f}")
            st.write(f"**p-valor:** {p_valor:.4f}")

            if p_valor < 0.05:
                st.success(
                    "Hay diferencias significativas entre al menos dos grupos."
                )
            else:
                st.warning(
                    "No se encontraron diferencias significativas entre los grupos."
                )
        else:
            st.warning(
                "Se necesitan al menos dos grupos con dos datos cada uno."
            )

    # Grafico de barras
    with Contenedor_B:
        st.subheader("Grafico de Barras: Media por grupo")

        resumen = df.groupby(Variable_x)[Variable_y].agg(
            ['mean', 'sem']
        ).reset_index()

        fig, ax = plt.subplots()

        x = resumen[Variable_x].astype(str)
        y = resumen['mean']
        yerr = resumen['sem'].fillna(0)

        ax.bar(
            x, y, yerr=yerr,
            capsize=5,
            color='skyblue',
            edgecolor='black'
        )

        ax.set_ylabel(f"Media de {Variable_y}")
        ax.set_xlabel(Variable_x)
        ax.set_title("Media por grupo con Error Estandar")
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()

        st.pyplot(fig)
        plt.close(fig)

Overwriting app.py
